Andrea Keiper and Alex Weinstein
DS4420 Final Project
April 16, 2026

MODEL 1: MANUALLY IMPLEMENTED MULTI-LAYER PERCEPTION

In [3]:
import pandas as pd
import numpy as np
import re

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [11]:
df = pd.read_csv("Boston_CrashDetails.csv", header=2)
df.head(5)

,Crash_Number,City_Town_Name,Crash_Date,Crash_Time,Crash_Severity,Maximum_Injury_Severity_Reported,Number_of_Vehicles,Total_Nonfatal_Injuries,Total_Fatal_Injuries,Manner_of_Collision,...,Ambient_Light,Weather_Condition,At_Roadway_Intersection,Distance_From_Nearest_Roadway_Intersection,Distance_From_Nearest_Milemarker,Distance_From_Nearest_Exit,Distance_From_Nearest_Landmark,Non_Motorist_Type,X_Cooordinate,Y_Cooordinate
0,2725539,BOSTON,01-Jan-2010,2:00 AM,Non-fatal injury,Non-fatal injury - Incapacitating,2,3,0,Rear-end,...,Daylight,"Sleet, hail (freezing rain or drizzle)",MORTON STREET / BLUE HILL AVENUE,MORTON STREET / BLUE HILL AVENUE,NaN,NaN,NaN,NaN,233688.296913,892724.750056
1,2576250,BOSTON,01-Jan-2010,2:02 AM,Not Reported,Not reported,2,0,0,Unknown,...,Dark - lighted roadway,Cloudy/Cloudy,BUNKER HILL STREET / ELM STREET,BUNKER HILL STREET / ELM STREET,NaN,NaN,NaN,NaN,236018.656211,903256.749810
2,2551708,BOSTON,01-Jan-2010,2:30 AM,Non-fatal injury,Non-fatal injury - Non-incapacitating,2,1,0,Angle,...,Dark - lighted roadway,Cloudy,NaN,COLUMBIA ROAD / BUTTONWOOD STREET,NaN,NaN,NaN,NaN,236609.587155,896844.662508
3,2560608,BOSTON,01-Jan-2010,2:32 AM,Property damage only (none injured),No injury,2,0,0,Rear-end,...,Dark - lighted roadway,Not Reported,MORTON STREET,MORTON STREET,NaN,NaN,NaN,NaN,NaN,NaN
4,2553095,BOSTON,01-Jan-2010,2:44 AM,Property damage only (none injured),No injury,1,0,0,Single vehicle crash,...,Dark - lighted roadway,Clear,NaN,Rte 90 W,Rte 90 W Milemarker 132.0,NaN,NaN,NaN,232572.239552,899870.482914


DATA CLEANING

In [13]:
df['Vehicle_Configuration'].unique()

array(['V1:(Passenger car) / V2:(Passenger car)',
       'V1:(Light truck(van, mini-van, pickup, sport utility))',
       'V1:(Passenger car)',
       'V1:(Light truck(van, mini-van, pickup, sport utility)) / V2:(Passenger car) / V3:(Passenger car)',
       nan, 'V1:(Passenger car) / V2:(Truck/trailer)',
       'V1:(Light truck(van, mini-van, pickup, sport utility)) / V2:(Passenger car)',
       'V1:(Light truck(van, mini-van, pickup, sport utility)) / V2:(Light truck(van, mini-van, pickup, sport utility))',
       'V1:(Passenger car) / V2:(Passenger car) / V3:(Light truck(van, mini-van, pickup, sport utility)) / V4:(Passenger car)',
       'V1:(Passenger car) / V2:(Light truck(van, mini-van, pickup, sport utility))',
       'V1:(Passenger car) / V2:(Light truck(van, mini-van, pickup, sport utility)) / V3:(Light truck(van, mini-van, pickup, sport utility))',
       'V2:(Light truck(van, mini-van, pickup, sport utility))',
       'V1:(Passenger car) / V2:(Bus (seats for 9-15 people, inc

In [ ]:
# 1. Clean vehicles, currently formatted to include ALL vehicles involved in the crash, I want to keep the largest vehicle involved
VEHICLE_RANK = {
    'Tractor/triples':                                  1,
    'Tractor/doubles':                                  1,
    'Tractor/semi-trailer':                             1,
    'Truck tractor (bobtail':                           2,
    'Single-unit truck (3-or-more axles':               2,
    'Unknown heavy truck, cannot classify':             2,
    'Single-unit truck (2-axle, 6-tires':               3,
    'Truck/trailer':                                    3,
    'Bus (seats for 16 or more, including driver':      4,
    'Bus (seats for 9-15 people, including driver':     4,
    'Motor home/recreational vehicle':                  5,
    'Light truck(van, mini-van, pickup, sport utility': 6,
    'Passenger car':                                    7,
    'Motorcycle':                                       8,
    'MOPED':                                            9,
    'All Terrain Vehicle (ATV':                         9,
    'Snowmobile':                                       9,
    'Low Speed Vehicle':                                9,
    'Registered farm equipment':                        9,
}

def largest_vehicle(val):
    # Extract all Vn types; return the one with the lowest rank number (= biggest vehicle)
    if pd.isna(val):
        return None
    matches = re.findall(r'V\d+:\((.+?)\)', str(val))
    if not matches:
        return None
    ranked = [(VEHICLE_RANK.get(m.strip(), 99), m.strip()) for m in matches]
    return min(ranked, key=lambda x: x[0])[1]
 
df['Vehicle_Type'] = df['Vehicle_Configuration'].apply(largest_vehicle)
 
# Consolidate into broad categories for one-hot encoding
VEHICLE_CATEGORY = {
    'Tractor/triples':                                  'Heavy Truck',
    'Tractor/doubles':                                  'Heavy Truck',
    'Tractor/semi-trailer':                             'Heavy Truck',
    'Truck tractor (bobtail':                           'Heavy Truck',
    'Single-unit truck (3-or-more axles':               'Heavy Truck',
    'Unknown heavy truck, cannot classify':             'Heavy Truck',
    'Single-unit truck (2-axle, 6-tires':               'Medium Truck',
    'Truck/trailer':                                    'Medium Truck',
    'Bus (seats for 16 or more, including driver':      'Bus',
    'Bus (seats for 9-15 people, including driver':     'Bus',
    'Motor home/recreational vehicle':                  'Light Truck/Van',
    'Light truck(van, mini-van, pickup, sport utility': 'Light Truck/Van',
    'Passenger car':                                    'Passenger Car',
    'Motorcycle':                                       'Motorcycle/Moped',
    'MOPED':                                            'Motorcycle/Moped',
    'All Terrain Vehicle (ATV':                         'Other',
    'Snowmobile':                                       'Other',
    'Low Speed Vehicle':                                'Other',
    'Registered farm equipment':                        'Other',
}

df['Largest_Vehicle'] = df['Vehicle_Type'].map(VEHICLE_CATEGORY)

Largest_Vehicle
Passenger Car       33597
Light Truck/Van     31877
Medium Truck         3077
Heavy Truck          2027
Bus                  2003
Motorcycle/Moped      510
Other                   6
Name: count, dtype: int64

In [ ]:
# 2. Clean intersection data
# Currently names the intersection if it's at an intersection, making it binary
df['At_Intersection'] = df['At_Roadway_Intersection'].notna().astype(int)


array([1, 0])

In [9]:
# 3. Clean weather
# Simplify: strip multi-condition combos (e.g. "Clear/Rain" -> "Clear")
df['Weather_Simple'] = df['Weather_Condition'].apply(
    lambda x: str(x).split('/')[0].strip() if pd.notna(x) else None
)
top_weather = df['Weather_Simple'].value_counts().nlargest(5).index
df['Weather_Simple'] = df['Weather_Simple'].apply(
    lambda x: x if x in top_weather else ('Other' if pd.notna(x) else None)
)

Prepare MLP

In [10]:
# TARGET: Binary Crash Severity
# 1 = any injury (fatal or non-fatal)
# 0 = property damage only / no injury
print(df['Crash_Severity'].unique())

# Remove unknown and nan
df = df[~df['Crash_Severity'].isin(['Not Reported', 'Unknown']) & df['Crash_Severity'].notna()].copy()

# Binary encode
injury_labels = {'Non-fatal injury', 'Fatal injury'}
df['severity_binary'] = df['Crash_Severity'].apply(
    lambda x: 1 if x in injury_labels else 0
)


['Non-fatal injury' 'Not Reported' 'Property damage only (none injured)'
 'Unknown' 'Fatal injury' nan]


In [30]:
CAT_FEATURES = ['Vehicle_Type', 'Road_Surface_Condition', 'Ambient_Light', 'Weather_Simple']
FEATURES = ['Vehicle_Type', 'Road_Surface_Condition', 'Ambient_Light', 'Weather_Simple', 'At_Intersection', 'severity_binary']
 
feature_df = df[FEATURES].dropna(subset=CAT_FEATURES).copy()
 
# Remove all ambiguous values
remove = {'Unknown', 'Not reported', 'Not Reported', 'Not Applicable', 'Other'}
mask = ~feature_df[CAT_FEATURES].apply(lambda col: col.isin(remove)).any(axis=1)
feature_df = feature_df[mask].copy()
feature_df = feature_df[feature_df[FEATURES].notna().all(axis=1)] 

print(f"Clean records: {len(feature_df):,}")
print(f"Class balance:\n{feature_df['severity_binary'].value_counts()}\n")

Clean records: 56,936
Class balance:
severity_binary
0    38795
1    18141
Name: count, dtype: int64



In [ ]:
# ONE-HOT ENCODE & SCALE (Homework 1 style)

encoded_df = pd.get_dummies(feature_df, columns=CAT_FEATURES, drop_first=True, dtype=int)
 
y = encoded_df['severity_binary'].to_numpy().astype(float)
X_raw = encoded_df.drop(columns=['severity_binary']).to_numpy().astype(float)
 
print(f"Feature matrix: {X_raw.shape}  ({X_raw.shape[1]} columns after one-hot)\n")
 
X_min = X_raw.min(axis=0)
X_max = X_raw.max(axis=0)
col_range = X_max - X_min
col_range[col_range == 0] = 1
X_scaled  = ((X_raw - X_min) / col_range).round(4)
 
# Add bias column
X_bias = np.concatenate([np.ones((X_scaled.shape[0], 1)), X_scaled], axis=1)

Feature matrix: (56936, 20)  (20 columns after one-hot)



In [33]:
# TRAIN / TEST SPLIT
# Are we allowed to use this
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_bias, y, test_size=0.3, random_state=42
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}\n")

Train: 39,855  |  Test: 17,081



In [34]:
# MLP SETUP 
# Input -> Hidden (ReLU) -> Output (Sigmoid)

np.random.seed(30)
 
d = X_train.shape[1]
h_size = 8
eta = 0.05
epochs = 300
 
W1 = np.random.randn(d, h_size) # (input_dim x hidden_size)
W2 = np.random.randn(h_size, 1) # (hidden_size x 1)

In [35]:
errors = []
 
for epoch in range(epochs):
    # Forward pass 
    # X_train: (n, d)  W1: (d, h)  -> H: (n, h)

    # ReLU hidden layer
    H = np.maximum(0, X_train @ W1)
    logit = H @ W2
    Y_hat = 1 / (1 + np.exp(-logit))
 
    y_col = y_train.reshape(-1, 1)
    error = Y_hat - y_col
 
    # Binary cross-entropy loss
    eps  = 1e-9
    loss = -np.mean(
        y_col * np.log(Y_hat + eps) + (1 - y_col) * np.log(1 - Y_hat + eps)
    )
    errors.append(float(loss))
 
    # Backward pass
    # W2 gradient: same as Homework 1 summed over samples
    dW2 = (1 / len(X_train)) * (H.T @ error)
 
    # W1 gradient: ReLU derivative * backprop signal
    # (n, h) heaviside of pre-activation
    relu_mask = (X_train @ W1 > 0).astype(float)  
    delta_h = (error @ W2.T) * relu_mask
    dW1 = (1 / len(X_train)) * (X_train.T @ delta_h)
 
    W1 -= eta * dW1
    W2 -= eta * dW2
 
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:>3}/{epochs} and Avg Loss: {loss:.4f}")
 
print()

Epoch  50/300 and Avg Loss: 0.6934
Epoch 100/300 and Avg Loss: 0.6622
Epoch 150/300 and Avg Loss: 0.6498
Epoch 200/300 and Avg Loss: 0.6429
Epoch 250/300 and Avg Loss: 0.6384
Epoch 300/300 and Avg Loss: 0.6351



In [ ]:
# EVALUATE

In [ ]:
# PLOT